<a href="https://colab.research.google.com/github/lilajabali/beginning-bioinformatics/blob/main/notebooks/IBD_Microbiome.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Set up Google Colab and Google Drive**

In [2]:

# Step 1: Connect Google Drive

from google.colab import drive
drive.mount('/content/drive')

# Step 2: Create the project folders

import os

PROJECT_DIR = "/content/drive/MyDrive/IBD_Microbiome_Project"

folders = [
    "01_Raw_Data",
    "02_Processed_Data",
    "03_Metadata",
    "04_Analysis_Results",
    "05_Figures",
    "06_Notebooks"
]

for folder in folders:
    os.makedirs(os.path.join(PROJECT_DIR, folder), exist_ok=True)

print("Google Drive connected!")
print("Project folders created successfully.")
print("Project location:", PROJECT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive connected!
Project folders created successfully.
Project location: /content/drive/MyDrive/IBD_Microbiome_Project


In [21]:
!pip install scikit-bio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 21.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [scikit-bio]


In [23]:
from skbio import TreeNode

# **Check files in drive**

In [4]:

import os

PROJECT_DIR = "/content/drive/MyDrive/IBD_Microbiome_Project"

folders_to_check = [
    "02_Processed_Data",
    "03_Metadata"
]

for folder in folders_to_check:
    folder_path = os.path.join(PROJECT_DIR, folder)
    print(f"\n--- {folder} ---")

    for filename in os.listdir(folder_path):
        print(filename)



--- 02_Processed_Data ---
reference-hit.biom

--- 03_Metadata ---
1629_20230206-074214.txt


# **Inspect Metadata**

In [5]:

import pandas as pd
import os

metadata_path = os.path.join(
    PROJECT_DIR,
    "03_Metadata",
    "1629_20230206-074214.txt"
)

metadata = pd.read_csv(metadata_path, sep="\t", dtype=str)

print("Metadata dimensions:", metadata.shape)

print("\nColumn names:")
print(metadata.columns.tolist())

print("\nFirst 5 rows:")
display(metadata.head())


Metadata dimensions: (683, 39)

Column names:
['sample_name', 'bmi', 'calprotectin', 'cd_behavior', 'cd_location', 'cd_resection', 'collection_timestamp', 'description', 'diagnosis_full', 'dna_extracted', 'env_biome', 'env_feature', 'env_material', 'env_package', 'fecal_date', 'geo_loc_name', 'host_body_habitat', 'host_body_product', 'host_body_site', 'host_common_name', 'host_scientific_name', 'host_subject_id', 'host_taxid', 'ibd_subtype', 'latitude', 'longitude', 'patientnumber', 'perianal_disease', 'physical_specimen_location', 'physical_specimen_remaining', 'qiita_study_id', 'sample_type', 'scientific_name', 'sex', 'study', 'study_id', 'timepoint', 'uc_extent', 'year_diagnosed']

First 5 rows:


,sample_name,bmi,calprotectin,cd_behavior,cd_location,cd_resection,collection_timestamp,description,diagnosis_full,dna_extracted,...,physical_specimen_remaining,qiita_study_id,sample_type,scientific_name,sex,study,study_id,timepoint,uc_extent,year_diagnosed
0,1629.SubjectIBD001,25,63.0,"non-stricturing, non-penetrating (b1)",colonic (l2),no,06/17/2010,SubjectIBD001,CD,true,...,false,1629,stool,human gut metagenome,male,IBD,1629,2,not applicable,1988
1,1629.SubjectIBD002,25,212.0,"non-stricturing, non-penetrating (b1)",colonic (l2),no,09/16/2010,SubjectIBD002,CD,true,...,false,1629,stool,human gut metagenome,male,IBD,1629,3,not applicable,1988
2,1629.SubjectIBD003,25,258.0,"non-stricturing, non-penetrating (b1)",colonic (l2),no,01/04/2011,SubjectIBD003,CD,true,...,false,1629,stool,human gut metagenome,male,IBD,1629,4,not applicable,1988
3,1629.SubjectIBD004,25,499.0,"non-stricturing, non-penetrating (b1)",colonic (l2),no,04/07/2011,SubjectIBD004,CD,true,...,false,1629,stool,human gut metagenome,male,IBD,1629,5,not applicable,1988
4,1629.SubjectIBD005,25,104.0,"non-stricturing, non-penetrating (b1)",colonic (l2),no,06/28/2011,SubjectIBD005,CD,true,...,false,1629,stool,human gut metagenome,male,IBD,1629,6,not applicable,1988


# **Count samples and participants**

In [6]:

# Count samples by diagnosis
print("Samples by diagnosis:")
print(metadata["diagnosis_full"].value_counts(dropna=False))

# Count unique participants by diagnosis
print("\nUnique participants by diagnosis:")
print(
    metadata.groupby("diagnosis_full")["host_subject_id"]
    .nunique()
)

# Check sample types
print("\nSample types:")
print(metadata["sample_type"].value_counts(dropna=False))


Samples by diagnosis:
diagnosis_full
UC    324
CD    251
HC     62
CC     36
LC     10
Name: count, dtype: int64

Unique participants by diagnosis:
diagnosis_full
CC    15
CD    49
HC     9
LC     4
UC    60
Name: host_subject_id, dtype: int64

Sample types:
sample_type
stool    683
Name: count, dtype: int64


In [7]:

# Install the BIOM package
!pip -q install biom-format

from biom import load_table
import os

# Locate the BIOM file
biom_path = os.path.join(
    PROJECT_DIR,
    "02_Processed_Data",
    "reference-hit.biom"
)

# Load the abundance table
table = load_table(biom_path)

# Extract sample IDs
biom_sample_ids = set(table.ids(axis="sample"))
metadata_sample_ids = set(metadata["sample_name"].dropna())

# Compare sample IDs
matching_ids = biom_sample_ids.intersection(metadata_sample_ids)

print("Samples in BIOM table:", len(biom_sample_ids))
print("Samples in metadata:", len(metadata_sample_ids))
print("Matching sample IDs:", len(matching_ids))
print("BIOM samples without metadata:", len(biom_sample_ids - metadata_sample_ids))
print("Metadata samples without BIOM data:", len(metadata_sample_ids - biom_sample_ids))

print("\nExample matching IDs:")
print(list(matching_ids)[:10])


Samples in BIOM table: 681
Samples in metadata: 683
Matching sample IDs: 681
BIOM samples without metadata: 0
Metadata samples without BIOM data: 2

Example matching IDs:
[np.str_('1629.SubjectIBD690'), np.str_('1629.SubjectIBD691'), np.str_('1629.SubjectIBD286'), np.str_('1629.SubjectIBD279'), np.str_('1629.SubjectIBD436'), np.str_('1629.SubjectIBD568'), np.str_('1629.SubjectIBD431'), np.str_('1629.SubjectIBD265'), np.str_('1629.SubjectIBD458'), np.str_('1629.SubjectIBD282')]


In [8]:

# Keep only samples present in both files
matched_metadata = metadata[
    metadata["sample_name"].isin(biom_sample_ids)
].copy()

# Keep our three study groups
groups = ["CD", "UC", "HC"]

eligible_metadata = matched_metadata[
    matched_metadata["diagnosis_full"].isin(groups)
].copy()

# Check unique participants in each group
print("Unique participants with matched BIOM data:")
print(
    eligible_metadata.groupby("diagnosis_full")["host_subject_id"]
    .nunique()
)

# Check whether any participant has more than one diagnosis
diagnosis_check = (
    eligible_metadata.groupby("host_subject_id")["diagnosis_full"]
    .nunique()
)

print("\nParticipants with multiple diagnoses:")
print((diagnosis_check > 1).sum())

# Check missing participant IDs
print("\nSamples with missing participant IDs:")
print(eligible_metadata["host_subject_id"].isna().sum())


Unique participants with matched BIOM data:
diagnosis_full
CD    49
HC     9
UC    60
Name: host_subject_id, dtype: int64

Participants with multiple diagnoses:
0

Samples with missing participant IDs:
0


# **Random selection of 29 samples**

In [9]:

import numpy as np
import pandas as pd
import os

# Set reproducible random seed
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

selected_rows = []

# Select participants and one sample per participant
for group, target_n in [("CD", 10), ("UC", 10), ("HC", 9)]:

    group_data = eligible_metadata[
        eligible_metadata["diagnosis_full"] == group
    ].copy()

    # Get unique participants
    participants = group_data["host_subject_id"].unique()

    # Randomly select participants
    chosen_participants = rng.choice(
        participants,
        size=target_n,
        replace=False
    )

    # Randomly select one sample for each participant
    for participant in chosen_participants:
        participant_samples = group_data[
            group_data["host_subject_id"] == participant
        ]

        chosen_sample = participant_samples.sample(
            n=1,
            random_state=int(rng.integers(0, 1_000_000))
        )

        selected_rows.append(chosen_sample)

# Combine selected samples
selected_metadata = pd.concat(selected_rows, ignore_index=True)

# Save selected metadata to Google Drive
selection_path = os.path.join(
    PROJECT_DIR,
    "03_Metadata",
    "selected_29_samples.csv"
)

selected_metadata.to_csv(selection_path, index=False)

# Display selection summary
print("Total selected samples:", len(selected_metadata))
print("\nSelected samples by group:")
print(selected_metadata["diagnosis_full"].value_counts())

print("\nUnique participants selected:")
print(selected_metadata["host_subject_id"].nunique())

print("\nSaved to:", selection_path)

display(
    selected_metadata[
        ["sample_name", "host_subject_id", "diagnosis_full"]
    ].sort_values("diagnosis_full")
)


Total selected samples: 29

Selected samples by group:
diagnosis_full
CD    10
UC    10
HC     9
Name: count, dtype: int64

Unique participants selected:
29

Saved to: /content/drive/MyDrive/IBD_Microbiome_Project/03_Metadata/selected_29_samples.csv


,sample_name,host_subject_id,diagnosis_full
0,1629.SubjectIBD531,1629:905,CD
9,1629.SubjectIBD393,1629:680,CD
7,1629.SubjectIBD051,1629:105,CD
6,1629.SubjectIBD344,1629:560,CD
5,1629.SubjectIBD234,1629:372,CD
8,1629.SubjectIBD116,1629:191,CD
3,1629.SubjectIBD314,1629:507,CD
2,1629.SubjectIBD046,1629:93,CD
1,1629.SubjectIBD342,1629:553,CD
4,1629.SubjectIBD233,1629:352,CD


Extract samples from BIOM file

In [10]:

# Get selected sample IDs
selected_ids = selected_metadata["sample_name"].tolist()

# Filter BIOM table to selected samples
filtered_table = table.filter(
    selected_ids,
    axis="sample",
    inplace=False
)

# Save filtered BIOM table
filtered_biom_path = os.path.join(
    PROJECT_DIR,
    "02_Processed_Data",
    "selected_29_samples.biom"
)

with open(filtered_biom_path, "w") as f:
    filtered_table.to_json("IBD_microbiome_analysis", direct_io=f)

print("Filtered BIOM table created!")
print("Number of samples:", filtered_table.shape[1])
print("Number of microbial features:", filtered_table.shape[0])
print("Saved to:", filtered_biom_path)


Filtered BIOM table created!
Number of samples: 29
Number of microbial features: 7589
Saved to: /content/drive/MyDrive/IBD_Microbiome_Project/02_Processed_Data/selected_29_samples.biom


# **Calculate relative abundance**

A: Correct the normalization

In [13]:

# Convert the original filtered BIOM table to a DataFrame
abundance_df = filtered_table.to_dataframe(dense=True)

# Calculate total counts for each sample
raw_sample_totals = abundance_df.sum(axis=0)

print("Raw sequencing totals for each sample:")
display(raw_sample_totals.to_frame(name="Total counts"))

print("\nSamples with zero counts:")
print(raw_sample_totals[raw_sample_totals == 0].index.tolist())

print("\nNumber of samples with zero counts:",
      (raw_sample_totals == 0).sum())


Raw sequencing totals for each sample:


,Total counts
1629.SubjectIBD021,35864.0
1629.SubjectIBD040,35576.0
1629.SubjectIBD129,359027.0
1629.SubjectIBD170,308689.0
1629.SubjectIBD116,367369.0
1629.SubjectIBD046,165966.0
1629.SubjectIBD051,78243.0
1629.SubjectIBD219,339216.0
1629.SubjectIBD220,353408.0
1629.SubjectIBD344,254685.0



Samples with zero counts:
[]

Number of samples with zero counts: 0


In [16]:

# Normalize the original raw abundance data
raw_totals = abundance_df.sum(axis=0)

relative_abundance_df = abundance_df.div(
    raw_totals,
    axis="columns"
)

# Verify each sample
check_totals = relative_abundance_df.sum(axis=0)

print("Number of samples:", relative_abundance_df.shape[1])
print("Minimum sample total:", check_totals.min())
print("Maximum sample total:", check_totals.max())
print("All samples sum to 1:", np.allclose(check_totals, 1.0))

# Save the normalized data
relative_csv_path = os.path.join(
    PROJECT_DIR,
    "02_Processed_Data",
    "relative_abundance_29_samples.csv"
)

relative_abundance_df.to_csv(relative_csv_path)

print("\nPandas relative abundance CSV saved successfully!")
print(relative_csv_path)


Number of samples: 29
Minimum sample total: 0.9999999999999998
Maximum sample total: 1.0000000000000002
All samples sum to 1: True

Pandas relative abundance CSV saved successfully!
/content/drive/MyDrive/IBD_Microbiome_Project/02_Processed_Data/relative_abundance_29_samples.csv


Microbial composition analysis

step 1:Inspect the BIOM taxonomy information

In [18]:

print("Total microbial features:", len(observation_ids))

print("\nFirst 5 feature IDs:")
for feature in observation_ids[:5]:
    print(feature)

print("\nBIOM table observation metadata:")
print(filtered_table.metadata(axis="observation"))


Total microbial features: 7589

First 5 feature IDs:
ACGGAGGATCCGAGCGTTATCCGGATTTATTGGGTTTAAAGGGAGCGTAGGCGGACGCTTAAGTCAGTTGTGAAAGTTTGCGGCTCAACC
ACGGAGGATCCGAGCGTTATCCGGATTTATTGGGTTTAAAGGGAGCGTAGGCGGATTGTTAAGTCAGTTGTGAAAGTTTGCGGCTCAACC
ACGGAGGATCCGAGCGTTATCCGGATTTATTGGGTTTAAAGGGAGCGTAGGTGGACTGGTAAGTCAGTTGTGAAAGTTTGCGGCTCAACC
ACGGAGGGTGCAAGCGTTAATCGGAATTACTGGGCGTAAAGCGCACGCAGGCGGTTTGTTAAGTCAGATGTGAAATCCCCGGGCTCAACC
ACGTAGGTGGCAAGCGTTGTCCGGAATTATTGGGCGTAAAGCGCGCGCAGGCGGATTGGTCAGTCTGTCTTAAAAGTTCGGGGCTTAACC

BIOM table observation metadata:
None


In [19]:
import os

tree_path = os.path.join(
    PROJECT_DIR,
    "02_Processed_Data",
    "insertion_tree.relabelled.tre"
)

print("Tree file found:", os.path.exists(tree_path))

if os.path.exists(tree_path):
    print("File size:", round(os.path.getsize(tree_path) / 1024, 2), "KB")

Tree file found: True
File size: 6771.44 KB


Check whether the tree matches your BIOM features

In [24]:
from skbio import TreeNode

# Load the phylogenetic tree
tree = TreeNode.read(tree_path)

# Get feature IDs from the BIOM table
biom_features = set(filtered_table.ids(axis="observation"))

# Get tip labels from the tree
tree_features = {tip.name for tip in tree.tips()}

# Compare them
print("BIOM features:", len(biom_features))
print("Tree tips:", len(tree_features))
print("Features shared:", len(biom_features.intersection(tree_features)))
print("BIOM features missing from tree:", len(biom_features - tree_features))

BIOM features: 7589
Tree tips: 211041
Features shared: 7589
BIOM features missing from tree: 0
